In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd

train_df = pd.read_csv("../data/processed/train_split.csv")
val_df   = pd.read_csv("../data/processed/val_split.csv")
test_df  = pd.read_csv("../data/processed/test_split_from_train.csv")

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print("Label distribution (train):")
print(train_df['label'].value_counts())

Train: 20,971 | Val: 2,621 | Test: 2,622
Label distribution (train):
label
1    13979
0     6992
Name: count, dtype: int64


In [2]:
# Schema mới: cột 'review' và 'label'
train_text = train_df["review"]
val_text   = val_df["review"]
test_text  = test_df["review"]

y_train = train_df["label"]
y_val   = val_df["label"]
y_test  = test_df["label"]

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# Preprocessing nhẹ cho TF-IDF: lowercase + bỏ URL
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_text_clean = train_text.apply(preprocess)
val_text_clean   = val_text.apply(preprocess)
test_text_clean  = test_text.apply(preprocess)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,  # giúp với corpus lớn hơn
)

X_train = vectorizer.fit_transform(train_text_clean)
X_val   = vectorizer.transform(val_text_clean)
X_test  = vectorizer.transform(test_text_clean)

print(f"Feature matrix: {X_train.shape}")

Feature matrix: (20971, 5000)


In [4]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # xử lý imbalanced data
    solver='lbfgs',
)

logreg.fit(X_train, y_train)
pred_val = logreg.predict(X_val)
print("Done")

Done


In [5]:
from sklearn.metrics import classification_report, confusion_matrix

print("=== Logistic Regression — Validation ===")
print(classification_report(y_val, pred_val, target_names=['negative', 'positive']))

pred_test_lr = logreg.predict(X_test)
print("=== Logistic Regression — Test ===")
print(classification_report(y_test, pred_test_lr, target_names=['negative', 'positive']))
print("Confusion matrix (test):")
print(confusion_matrix(y_test, pred_test_lr))

=== Logistic Regression — Validation ===
              precision    recall  f1-score   support

    negative       0.81      0.89      0.85       874
    positive       0.94      0.90      0.92      1747

    accuracy                           0.89      2621
   macro avg       0.88      0.89      0.88      2621
weighted avg       0.90      0.89      0.90      2621

=== Logistic Regression — Test ===
              precision    recall  f1-score   support

    negative       0.82      0.89      0.85       874
    positive       0.94      0.90      0.92      1748

    accuracy                           0.90      2622
   macro avg       0.88      0.89      0.88      2622
weighted avg       0.90      0.90      0.90      2622

Confusion matrix (test):
[[ 776   98]
 [ 176 1572]]


In [6]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm = LinearSVC(class_weight='balanced', max_iter=2000)
svm.fit(X_train, y_train)

svm_pred_val  = svm.predict(X_val)
svm_pred_test = svm.predict(X_test)

print("=== SVM — Validation ===")
print(classification_report(y_val, svm_pred_val, target_names=['negative', 'positive']))

print("=== SVM — Test ===")
print(classification_report(y_test, svm_pred_test, target_names=['negative', 'positive']))
print("Confusion matrix (test):")
print(confusion_matrix(y_test, svm_pred_test))

=== SVM — Validation ===
              precision    recall  f1-score   support

    negative       0.80      0.86      0.83       874
    positive       0.93      0.89      0.91      1747

    accuracy                           0.88      2621
   macro avg       0.86      0.88      0.87      2621
weighted avg       0.89      0.88      0.88      2621

=== SVM — Test ===
              precision    recall  f1-score   support

    negative       0.80      0.88      0.83       874
    positive       0.93      0.89      0.91      1748

    accuracy                           0.88      2622
   macro avg       0.87      0.88      0.87      2622
weighted avg       0.89      0.88      0.89      2622

Confusion matrix (test):
[[ 765  109]
 [ 194 1554]]


In [7]:
# ── Tổng hợp so sánh cả 3 model trên test set ───────────────────
from sklearn.metrics import accuracy_score, f1_score

models = {
    'Logistic Regression': pred_test_lr,
    'SVM (LinearSVC)':     svm_pred_test,
}

print(f"{'Model':<25} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 48)
for name, preds in models.items():
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='macro')
    print(f"{name:<25} {acc:>10.4f} {f1:>10.4f}")
print("-" * 48)
print(f"{'PhoBERT (reference)':<25} {'0.9211':>10} {'0.9116':>10}")

Model                       Accuracy   Macro F1
------------------------------------------------
Logistic Regression           0.8955     0.8849
SVM (LinearSVC)               0.8844     0.8729
------------------------------------------------
PhoBERT (reference)           0.9211     0.9116
